# 02 — Feature Engineering
**TF-IDF Vectorization + BERT Embeddings**

This notebook:

1. Builds TF-IDF feature matrices (fit on train, transform test)
2. Generates BERT sentence embeddings via `sentence-transformers`
3. Saves all matrices + the fitted vectorizer / model to `../data/features/`

## Imports & Config

In [2]:
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
import joblib
from tqdm.auto import tqdm

# TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer

# BERT embeddings
from sentence_transformers import SentenceTransformer

In [25]:
import os
from google.colab import drive



drive.mount('/content/drive')
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/SmartReviewAnalyzer'




#Paths
PROCESSED_DIR = os.path.join(DRIVE_PROJECT_ROOT, 'data/processed')
FEATURES_DIR  = os.path.join(DRIVE_PROJECT_ROOT, 'data/features')

TRAIN_CSV = os.path.join(PROCESSED_DIR, 'preprocessed_train.csv')
TEST_CSV  = os.path.join(PROCESSED_DIR, 'preprocessed_test.csv')

os.makedirs(FEATURES_DIR, exist_ok=True)

#TF-IDF hyper-parameters
TFIDF_MAX_FEATURES   = 50_000
TFIDF_NGRAM_RANGE    = (1, 2)
TFIDF_MIN_DF         = 2
TFIDF_SUBLINEAR_TF   = True

#BERT model
BERT_MODEL_NAME = 'all-MiniLM-L6-v2'
BERT_BATCH_SIZE = 64

print('Config ready (Google Drive Connected).')
print(f'  Features dir : {FEATURES_DIR}')
print(f'  TF-IDF vocab : {TFIDF_MAX_FEATURES:,} max features')
print(f'  BERT model   : {BERT_MODEL_NAME}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Config ready (Google Drive Connected).
  Features dir : /content/drive/MyDrive/SmartReviewAnalyzer/data/features
  TF-IDF vocab : 50,000 max features
  BERT model   : all-MiniLM-L6-v2


## Load Preprocessed Data

In [4]:
df_train = pd.read_csv(TRAIN_CSV)
df_test  = pd.read_csv(TEST_CSV)

print(f'Train : {df_train.shape}')
print(f'Test  : {df_test.shape}')
df_train.head(2)

Train : (519575, 4)
Test  : (39972, 4)


,binary_label,text,cleaned_text_tfidf,cleaned_text_bert
0,1,dr. goldberg offers everything i look for in a...,dr goldberg offer everything look general prac...,dr. goldberg offers everything i look for in a...
1,0,"Unfortunately, the frustration of being Dr. Go...",unfortunately frustration dr goldberg patient ...,"Unfortunately, the frustration of being Dr. Go..."


In [5]:
# Separate text columns and labels
train_text_tfidf = df_train['cleaned_text_tfidf'].fillna('').tolist()
test_text_tfidf  = df_test['cleaned_text_tfidf'].fillna('').tolist()

train_text_bert  = df_train['cleaned_text_bert'].fillna('').tolist()
test_text_bert   = df_test['cleaned_text_bert'].fillna('').tolist()

y_train = df_train['binary_label'].values
y_test  = df_test['binary_label'].values

print(f'Train texts (TF-IDF) : {len(train_text_tfidf):,}')
print(f'Test  texts (TF-IDF) : {len(test_text_tfidf):,}')
print(f'Train texts (BERT)   : {len(train_text_bert):,}')
print(f'Test  texts (BERT)   : {len(test_text_bert):,}')

Train texts (TF-IDF) : 519,575
Test  texts (TF-IDF) : 39,972
Train texts (BERT)   : 519,575
Test  texts (BERT)   : 39,972


---
## TF-IDF Vectorization

### Initialize TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer(
    max_features = TFIDF_MAX_FEATURES,
    ngram_range  = TFIDF_NGRAM_RANGE,
    min_df       = TFIDF_MIN_DF,
    sublinear_tf = TFIDF_SUBLINEAR_TF,
    strip_accents= 'unicode',
    analyzer     = 'word',
    token_pattern= r'\b[a-zA-Z][a-zA-Z]+\b',
)

print('TfidfVectorizer initialised.')
print(f'  max_features  : {TFIDF_MAX_FEATURES:,}')
print(f'  ngram_range   : {TFIDF_NGRAM_RANGE}')
print(f'  min_df        : {TFIDF_MIN_DF}')
print(f'  sublinear_tf  : {TFIDF_SUBLINEAR_TF}')

TfidfVectorizer initialised.
  max_features  : 50,000
  ngram_range   : (1, 2)
  min_df        : 2
  sublinear_tf  : True


### Fit on Training Data

In [ ]:
print('Fitting TF-IDF on training corpus ...')
X_train_tfidf = tfidf.fit_transform(train_text_tfidf)

print(f'Vocabulary size  : {len(tfidf.vocabulary_):,}')
print(f'Train matrix     : {X_train_tfidf.shape}  (sparse float32)')
print(f'Non-zero entries : {X_train_tfidf.nnz:,}')

Fitting TF-IDF on training corpus ...
Vocabulary size  : 50,000
Train matrix     : (519575, 50000)  (sparse float32)
Non-zero entries : 40,631,495


### Transform Validation / Test Data

In [ ]:
print('Transforming test corpus ...')
X_test_tfidf = tfidf.transform(test_text_tfidf)

print(f'Test matrix : {X_test_tfidf.shape}  (sparse float32)')

# Sanity check — top tokens by mean TF-IDF weight on training set
feature_names = np.array(tfidf.get_feature_names_out())
mean_scores   = np.asarray(X_train_tfidf.mean(axis=0)).ravel()
top10_idx     = mean_scores.argsort()[::-1][:10]

print('\nTop-10 features by mean TF-IDF weight (train):')
for rank, idx in enumerate(top10_idx, 1):
    print(f'  {rank:2d}. {feature_names[idx]:<25s}  {mean_scores[idx]:.5f}')

Transforming test corpus ...
Test matrix : (39972, 50000)  (sparse float32)

Top-10 features by mean TF-IDF weight (train):
   1. not                        0.02973
   2. food                       0.01979
   3. place                      0.01968
   4. go                         0.01925
   5. good                       0.01882
   6. get                        0.01871
   7. great                      0.01678
   8. service                    0.01571
   9. time                       0.01530
  10. very                       0.01517


### Save TF-IDF Features

In [ ]:
# Sparse matrices → .npz (preserves sparsity, fast I/O)
tfidf_train_path = os.path.join(FEATURES_DIR, 'X_train_tfidf.npz')
tfidf_test_path  = os.path.join(FEATURES_DIR, 'X_test_tfidf.npz')
tfidf_model_path = os.path.join(FEATURES_DIR, 'tfidf_vectorizer.joblib')
labels_path      = os.path.join(FEATURES_DIR, 'labels.npz')

sp.save_npz(tfidf_train_path, X_train_tfidf)
sp.save_npz(tfidf_test_path,  X_test_tfidf)
joblib.dump(tfidf, tfidf_model_path)
np.savez(labels_path, y_train=y_train, y_test=y_test)

print('TF-IDF artefacts saved:')
for p in [tfidf_train_path, tfidf_test_path, tfidf_model_path, labels_path]:
    size_mb = os.path.getsize(p) / 1_048_576
    print(f'  {p}  ({size_mb:.2f} MB)')

TF-IDF artefacts saved:
  /content/drive/MyDrive/SmartReviewAnalyzer/data/features/X_train_tfidf.npz  (371.98 MB)
  /content/drive/MyDrive/SmartReviewAnalyzer/data/features/X_test_tfidf.npz  (28.25 MB)
  /content/drive/MyDrive/SmartReviewAnalyzer/data/features/tfidf_vectorizer.joblib  (1.90 MB)
  /content/drive/MyDrive/SmartReviewAnalyzer/data/features/labels.npz  (4.27 MB)


---
## BERT Embeddings (Option B — Recommended)

Using [`sentence-transformers`](https://www.sbert.net/) with **`all-MiniLM-L6-v2`**:
- 6-layer MiniLM distilled from BERT — ~80 MB download
- **384-dimensional** embeddings, L2-normalised
- Excellent quality/speed tradeoff for sentiment classification


### Load BERT Model

In [6]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {device}')

print(f'Loading {BERT_MODEL_NAME} ...')
bert_model = SentenceTransformer(BERT_MODEL_NAME, device=device)

embedding_dim = bert_model.get_sentence_embedding_dimension()
print(f'Model loaded  —  embedding dim : {embedding_dim}')

Device : cuda
Loading all-MiniLM-L6-v2 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded  —  embedding dim : 384


/tmp/ipykernel_2587/975309461.py:9: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedding_dim = bert_model.get_sentence_embedding_dimension()


### Convert Text → Embeddings

In [7]:
print(f'Encoding {len(train_text_bert):,} training examples ...')
X_train_bert = bert_model.encode(
    train_text_bert,
    batch_size           = BERT_BATCH_SIZE,
    show_progress_bar    = True,
    normalize_embeddings = True,
    convert_to_numpy     = True,
)

print(f'Train embedding matrix : {X_train_bert.shape}  dtype={X_train_bert.dtype}')

Encoding 519,575 training examples ...


Batches:   0%|          | 0/8119 [00:00<?, ?it/s]

Train embedding matrix : (519575, 384)  dtype=float32


In [8]:
print(f'Encoding {len(test_text_bert):,} test examples ...')
X_test_bert = bert_model.encode(
    test_text_bert,
    batch_size           = BERT_BATCH_SIZE,
    show_progress_bar    = True,
    normalize_embeddings = True,
    convert_to_numpy     = True,
)

print(f'Test  embedding matrix : {X_test_bert.shape}  dtype={X_test_bert.dtype}')

# Sanity — L2 norms should be ~1.0 after normalisation
norms = np.linalg.norm(X_train_bert[:100], axis=1)
print(f'\nEmbedding L2 norms (first 100 train rows):')
print(f'  mean={norms.mean():.4f}  min={norms.min():.4f}  max={norms.max():.4f}')

Encoding 39,972 test examples ...


Batches:   0%|          | 0/625 [00:00<?, ?it/s]

Test  embedding matrix : (39972, 384)  dtype=float32

Embedding L2 norms (first 100 train rows):
  mean=1.0000  min=1.0000  max=1.0000


### Save Embedding Features

In [29]:
MY_FEATURES_DIR = '/content/drive/MyDrive/SmartReviewAnalyzer_features'
os.makedirs(MY_FEATURES_DIR, exist_ok=True)
# BERT
np.save(f'{MY_FEATURES_DIR}/X_train_bert.npy', X_train_bert)
np.save(f'{MY_FEATURES_DIR}/X_test_bert.npy',  X_test_bert)
with open(f'{MY_FEATURES_DIR}/bert_model_name.txt', 'w') as f:
    f.write(BERT_MODEL_NAME)

